# Exposure to soybean and palm oil prices by province (GAEZ)

**Goal.** Same exposure as `exposure_gaez.ipynb`, but for each **province** (first administrative level:
states in Brazil, provinces in Indonesia, regions in Ghana...) of our 62 countries, and a province x year
panel of forest loss to go with it.

**Why provinces.** A national average hides where the forest meets the crops. In Brazil, the Amazon as a
whole is only moderately suitable for soybeans, but the states on the agricultural frontier are not.
With provinces we can also add **country x year fixed effects**, which absorb everything national
(policies such as PPCDAm or the Soy Moratorium, exchange rates, governance, crises): only provinces of
the same country in the same year are compared.

For each province $j$ and crop $c$:

$$E_{jc} = \frac{\sum_{k \in j} \tfrac{SI_{ck}}{10\,000} \times F_k}{\sum_{k \in j} F_k}$$

with $k$ the 5' grid cells of the province, $SI$ the GAEZ v4 suitability index and $F$ the tree cover in
2000 (km² with canopy > 30%).

**Outcome.** DeDuCE (deforestation attributed to oil palm or soybeans) only exists by country. By
province we use the Global Forest Watch tree cover loss attributed to **permanent agriculture**, the
same variable as our country-level outcome (`agri_loss_rate`).

**Data** (see `docs/data_log.md`):
- `raw/forest/global.xlsx`: Global Forest Watch, sheets "Subnational 1 tree cover loss" and "Subnational 1 drivers"
- `raw/boundaries/gadm41_admin1_sample.gpkg`: GADM 4.1 province borders (the version GFW uses), built by `src/build_gadm_admin1.py`
- `processed/forest2000_5arcmin.csv`: Hansen tree cover 2000 on the 5' grid, built by `src/build_forest2000_grid.py`
- `raw/gaez/sxHr_soy.tif`, `raw/gaez/sxHr_olp.tif`: GAEZ v4 suitability rasters
- `raw/boundaries/ne_10m_admin_0_countries.zip`: Natural Earth country borders (only to recognise sea cells)
- `processed/commodity_prices_annual.csv`: World Bank Pink Sheet real prices

**Outputs:** `processed/exposure_gaez_province.csv`, `processed/province_panel.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import from_origin

# Data is loaded from GitHub with URL links, so the notebook runs on any computer
BASE_URL = "https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/exposure/data/"
FIGURES = "../../figures/"
PROCESSED_OUT = "../../data/processed/"

CROPS = {"soy": "soybeans", "olp": "palm_oil"}   # GAEZ code -> name used in the columns
MIN_FOREST_HA = 10_000   # provinces with less forest in 2000 are left out (about one grid cell)

# 5 arc-minute grid of GAEZ: 2160 rows x 4320 columns, row 0 at 90N, column 0 at 180W
CELLS_PER_DEG = 12
N_ROWS, N_COLS = 180 * CELLS_PER_DEG, 360 * CELLS_PER_DEG
GRID = from_origin(-180, 90, 1 / CELLS_PER_DEG, 1 / CELLS_PER_DEG)

SOURCE = "Sources: FAO/IIASA GAEZ v4; Hansen et al. (2013) GFC-2024-v1.12; GADM 4.1; Global Forest Watch"


def add_source(fig, text=SOURCE):
    # Write the data source under the chart
    fig.text(0.01, 0.005, text, fontsize=8, color="#666666", ha="left", va="bottom")

## 1. Load the data and match the province names

In [ ]:
sample = pd.read_csv(BASE_URL + "processed/shock_panel.csv")
countries = sample[["iso3", "country", "region"]].drop_duplicates("iso3").reset_index(drop=True)
print(len(countries), "countries")

# Global Forest Watch by province (canopy threshold 30%, as for our country panel)
gfw = pd.ExcelFile(BASE_URL + "raw/forest/global.xlsx")
loss = pd.read_excel(gfw, sheet_name="Subnational 1 tree cover loss")
drivers = pd.read_excel(gfw, sheet_name="Subnational 1 drivers")
loss = loss[loss["threshold"] == 30].merge(countries, on="country")
drivers = drivers[drivers["threshold"] == 30].merge(countries, on="country")

# A few provinces appear on several rows (e.g. three Indian states): add them up
loss_cols = [c for c in loss.columns if c.startswith("tc_loss_ha_")]
loss = (loss.groupby(["iso3", "country", "region", "subnational1"], as_index=False)
            [["area_ha", "extent_2000_ha"] + loss_cols].sum())
print(f"GFW: {len(loss):,} provinces in {loss.iso3.nunique()} countries")

provinces = gpd.read_file(BASE_URL + "raw/boundaries/gadm41_admin1_sample.gpkg")
print(f"GADM: {len(provinces):,} provinces")

GFW names its provinces with GADM 4.1, so the names should match. We compare them without spaces and
capitals (GFW writes "OrangeWalk" where GADM writes "Orange Walk" in Belize), and list the provinces
that still do not match.

In [ ]:
def name_key(s):
    return s.str.lower().str.replace(" ", "", regex=False)


loss["name_key"] = name_key(loss["subnational1"])
provinces["name_key"] = name_key(provinces["name_1"])
assert not loss.duplicated(["iso3", "name_key"]).any() and not provinces.duplicated(["iso3", "name_key"]).any()

names = loss[["iso3", "name_key", "subnational1", "extent_2000_ha"]].merge(
    provinces[["iso3", "name_key", "name_1", "gid_1"]], on=["iso3", "name_key"], how="outer", indicator=True)
print(names["_merge"].value_counts().rename({"both": "matched", "left_only": "GFW only", "right_only": "GADM only"}))
names[names["_merge"] != "both"].sort_values("iso3")[["iso3", "subnational1", "name_1", "extent_2000_ha"]]

In [ ]:
# Keep the matched provinces; gid_1 (GADM code) becomes the province key
key = names.loc[names["_merge"] == "both", ["iso3", "subnational1", "gid_1"]]
loss = loss.merge(key, on=["iso3", "subnational1"])
drivers = drivers.merge(key, on=["iso3", "subnational1"])
provinces = provinces[provinces["gid_1"].isin(key["gid_1"])].reset_index(drop=True)
print(f"{len(loss):,} provinces kept, {loss.extent_2000_ha.sum() / sample.drop_duplicates('iso3').extent_2000_ha.sum():.1%} of the forest of the 62 countries")

## 2. Put every grid cell in a province

A cell goes to the province that contains its centre. A coastal cell whose centre falls in the sea goes
to a province it touches (otherwise small islands lose a large part of their forest). We use the Natural
Earth land mask to recognise sea cells, so that a cell of a neighbouring country outside our sample is
never given to one of our provinces.

In [ ]:
forest = pd.read_csv(BASE_URL + "processed/forest2000_5arcmin.csv")

shapes = [(geom, k + 1) for k, geom in enumerate(provinces.geometry)]   # 0 = no province
centre = features.rasterize(shapes, out_shape=(N_ROWS, N_COLS), transform=GRID, fill=0, dtype="int32")
touched = features.rasterize(shapes, out_shape=(N_ROWS, N_COLS), transform=GRID, fill=0, dtype="int32",
                             all_touched=True)

world = gpd.read_file(BASE_URL + "raw/boundaries/ne_10m_admin_0_countries.zip")
land = features.rasterize(((g, 1) for g in world.geometry), out_shape=(N_ROWS, N_COLS), transform=GRID,
                          fill=0, dtype="uint8")
province_id = np.where(centre > 0, centre, np.where(land == 0, touched, 0))

gid_of_id = np.r_[[None], provinces["gid_1"].values]
forest["gid_1"] = gid_of_id[province_id[forest.row, forest.col]]
forest = forest[forest["gid_1"].notna()].reset_index(drop=True)
print(f"{len(forest):,} forest cells in our provinces")

**Check.** Summed by province, the grid should give back the GFW tree cover in 2000. The 5' cells
(about 86 km²) are large for small provinces, so we expect more noise there than by country.

In [ ]:
check = loss.set_index("gid_1")[["iso3", "subnational1", "extent_2000_ha"]].copy()
check["grid_ha"] = forest.groupby("gid_1").forest2000_km2.sum().reindex(check.index).fillna(0) * 100
check["ratio"] = check.grid_ha / check.extent_2000_ha
big = check[check.extent_2000_ha >= MIN_FOREST_HA]

print(f"all provinces: grid / GFW = {check.grid_ha.sum() / check.extent_2000_ha.sum():.3f}")
print(f"{len(big)} provinces with at least {MIN_FOREST_HA:,} ha of forest:")
print(f"  median ratio {big.ratio.median():.3f}, "
      f"{(big.ratio.between(0.9, 1.1)).mean():.0%} within 10%, {(big.ratio.between(0.75, 1.25)).mean():.0%} within 25%")
print("Largest gaps among the provinces with more than 100,000 ha of forest:")
large = big[big.extent_2000_ha >= 100_000]
large.loc[(large.ratio - 1).abs().sort_values(ascending=False).index].head(8).round(3)

## 3. Exposure of each province

Main measure: average suitability of the forest of 2000 (`suit_forest_*`). Variant: average suitability
of all the land of the province (`suit_land_*`). We keep the provinces with at least 10,000 ha of forest
in 2000 (`MIN_FOREST_HA`): below that, a province holds about one grid cell and its loss rate is very
noisy. Each measure is then standardised across these provinces (mean 0, standard deviation 1).

In [ ]:
suit = {}
for crop in CROPS:
    with rasterio.open(BASE_URL + f"raw/gaez/sxHr_{crop}.tif") as src:
        assert src.shape == (N_ROWS, N_COLS) and src.transform.almost_equals(GRID)
        a = src.read(1).astype(float)
        a[a == src.nodata] = np.nan
        suit[crop] = a / 10000

# area (km2) of each row of 5' cells
R = 6371.0072
lat_top = 90 - np.arange(N_ROWS) / CELLS_PER_DEG
cell_km2 = R**2 * np.deg2rad(1 / CELLS_PER_DEG) * (np.sin(np.deg2rad(lat_top)) - np.sin(np.deg2rad(lat_top - 1 / CELLS_PER_DEG)))

land_cells = pd.DataFrame(np.argwhere(centre > 0), columns=["row", "col"])
land_cells["gid_1"] = gid_of_id[centre[land_cells.row, land_cells.col]]
land_cells["area_km2"] = cell_km2[land_cells.row]

for crop in CROPS:
    forest[crop] = suit[crop][forest.row, forest.col]
    land_cells[crop] = suit[crop][land_cells.row, land_cells.col]


def weighted_mean(df, value, weight):
    ok = df[value].notna()
    w = df.loc[ok, weight]
    return (df.loc[ok, value] * w).groupby(df.loc[ok, "gid_1"]).sum() / w.groupby(df.loc[ok, "gid_1"]).sum()


exposure = loss[["iso3", "country", "region", "gid_1", "subnational1", "extent_2000_ha"]].rename(
    columns={"subnational1": "province"}).set_index("gid_1")
for crop, name in CROPS.items():
    exposure[f"suit_forest_{name}"] = weighted_mean(forest, crop, "forest2000_km2")
    exposure[f"suit_land_{name}"] = weighted_mean(land_cells, crop, "area_km2")

kept = exposure["extent_2000_ha"].ge(MIN_FOREST_HA) & exposure.filter(like="suit_").notna().all(axis=1)
print(f"{kept.sum()} provinces kept out of {len(exposure)}, in {exposure.loc[kept, 'iso3'].nunique()} countries; "
      f"they hold {exposure.loc[kept, 'extent_2000_ha'].sum() / exposure.extent_2000_ha.sum():.1%} of the forest")
exposure = exposure[kept].copy()

for c in [c for c in exposure.columns if c.startswith("suit_")]:
    exposure["E_" + c.removeprefix("suit_")] = (exposure[c] - exposure[c].mean()) / exposure[c].std()

exposure = exposure.reset_index()
exposure.to_csv(PROCESSED_OUT + "exposure_gaez_province.csv", index=False)
print("saved exposure_gaez_province.csv", exposure.shape)
exposure.filter(like="suit_").describe().round(3)

**How much of the variation is between provinces of the same country?** This is the variation left once
country x year fixed effects are in the regression.

In [ ]:
for c in ["suit_forest_soybeans", "suit_forest_palm_oil"]:
    within = exposure[c] - exposure.groupby("iso3")[c].transform("mean")
    print(f"{c}: {within.var() / exposure[c].var():.0%} of the variance is within countries")

In [ ]:
# Brazil: the 27 states (26 states and the Federal District)
cols = ["province", "extent_2000_ha", "suit_forest_soybeans", "suit_forest_palm_oil"]
exposure.loc[exposure.iso3 == "BRA", cols].sort_values("suit_forest_soybeans", ascending=False).round(3)

In [ ]:
# Map of the main measure (forest of 2000) by province
gmap = provinces.merge(exposure[["gid_1", "suit_forest_soybeans", "suit_forest_palm_oil"]], on="gid_1", how="left")

fig, axes = plt.subplots(2, 1, figsize=(12, 9.5))
for ax, (col, title) in zip(axes, [("suit_forest_soybeans", "Soybeans"), ("suit_forest_palm_oil", "Palm oil")]):
    gmap[gmap[col].isna()].plot(ax=ax, color="#e8e8e6", edgecolor="white", linewidth=0.2)
    gmap[gmap[col].notna()].plot(ax=ax, column=col, cmap="Blues", vmin=0, vmax=0.8, edgecolor="white",
                                 linewidth=0.2, legend=True,
                                 legend_kwds={"label": "average suitability of the forest of 2000", "shrink": 0.7})
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlim(-120, 180); ax.set_ylim(-45, 40)
    ax.set_axis_off()
fig.suptitle("GAEZ suitability of the forest of 2000, by province (grey: less than 10,000 ha of forest)",
             x=0.01, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0.02, 1, 0.97))
add_source(fig)
fig.savefig(FIGURES + "exposure_gaez_province_map.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Province x year panel

Tree cover loss (all drivers) and tree cover loss attributed to permanent agriculture, 2001-2025, as a
percentage of the tree cover in 2000, the same definition as `agri_loss_rate` in our country panel.
Many province-years have no agricultural loss at all, so the log is not defined there: we also give
`asinh`, which is close to the log for large values and equals 0 at 0.
We add the lagged log real world prices of soybeans and palm oil (World Bank Pink Sheet).

In [ ]:
years = range(2001, 2026)
panel = (loss.melt(id_vars=["gid_1", "iso3", "country", "region", "subnational1", "extent_2000_ha"],
                   value_vars=loss_cols, var_name="year", value_name="loss_ha")
             .rename(columns={"subnational1": "province"}))
panel["year"] = panel["year"].str.removeprefix("tc_loss_ha_").astype(int)

agri = (drivers[drivers["driver"] == "Permanent agriculture"]
        .groupby(["gid_1", "year"], as_index=False)["tc_loss_ha"].sum()
        .rename(columns={"tc_loss_ha": "agri_loss_ha"}))
panel = panel.merge(agri, on=["gid_1", "year"], how="left", validate="one_to_one")
panel["agri_loss_ha"] = panel["agri_loss_ha"].fillna(0)   # no row = no loss attributed to agriculture

panel = panel[panel["gid_1"].isin(exposure["gid_1"])]
panel["loss_rate"] = 100 * panel["loss_ha"] / panel["extent_2000_ha"]
panel["agri_loss_rate"] = 100 * panel["agri_loss_ha"] / panel["extent_2000_ha"]
panel["asinh_agri_loss_rate"] = np.arcsinh(panel["agri_loss_rate"])
panel["log_agri_loss_rate"] = np.log(panel["agri_loss_rate"].where(panel["agri_loss_rate"] > 0))

# lagged log real world prices
prices = pd.read_csv(BASE_URL + "processed/commodity_prices_annual.csv")
for commodity, name in [("Soybeans", "soybeans"), ("Palm oil", "palm_oil")]:
    p = prices.loc[prices["commodity"] == commodity, ["year", "log_price_real"]]
    p = p.assign(year=p["year"] + 1).rename(columns={"log_price_real": f"log_p_{name}_lag1"})
    panel = panel.merge(p, on="year", how="left", validate="many_to_one")

panel = panel.merge(exposure[["gid_1"] + [c for c in exposure.columns if c.startswith("E_")]], on="gid_1",
                    validate="many_to_one")
panel = panel.sort_values(["gid_1", "year"]).reset_index(drop=True)

assert len(panel) == len(exposure) * len(years)
assert panel.filter(like="log_p_").notna().all().all()
panel.to_csv(PROCESSED_OUT + "province_panel.csv", index=False)
print(f"saved province_panel.csv: {len(panel):,} rows = {panel.gid_1.nunique()} provinces x {len(years)} years")

In [ ]:
# Consistency with the country panel: summed over provinces, agricultural loss should be close to the country figure
by_country = panel.groupby(["iso3", "year"])["agri_loss_ha"].sum().rename("provinces")
both = sample.set_index(["iso3", "year"])["agri_loss_ha"].rename("country").to_frame().join(by_country, how="inner")
print(f"agricultural loss, sum of our provinces / country total: {both.provinces.sum() / both.country.sum():.3f}")
print("(provinces with less than 10,000 ha of forest are left out: they hold almost no agricultural loss)")

summary = panel.groupby("region").agg(
    provinces=("gid_1", "nunique"),
    mean_agri_loss_rate=("agri_loss_rate", "mean"),
    median_agri_loss_rate=("agri_loss_rate", "median"),
    share_zero=("agri_loss_rate", lambda s: (s == 0).mean()))
summary.loc["All"] = [panel.gid_1.nunique(), panel.agri_loss_rate.mean(), panel.agri_loss_rate.median(),
                      (panel.agri_loss_rate == 0).mean()]
summary.round(3)

## 5. Summary

*To write once the numbers above are checked.* Points to cover:
- how many provinces are kept and how much of the forest they hold (sections 1 and 3)
- how close the grid is to GFW by province, and where it is not (section 2)
- how much of the exposure varies within countries, which is what country x year fixed effects keep (section 3)
- the Brazilian states: do the frontier states stand out? (section 3)
- share of zeros in the agricultural loss rate and the choice between log and asinh (section 4)

**Next step:** regression of `asinh_agri_loss_rate` on `E_forest_palm_oil x log_p_palm_oil_lag1` (and the
same for soybeans) with province fixed effects and country x year fixed effects, standard errors
clustered by country.